In [1]:
import numpy as np
from namap_main import load_par_file
import src.loaddata as ld
import src.detector as tod
import src.mapmaker as mp
import src.pointing as pt  
import copy
from astropy import wcs 
import astropy.table as tb
import h5py
import argparse
import ast
import sys
from astropy.table import Table


#for debugging purpose only
from IPython import embed

#for profilling purpose only
import tracemalloc
import time

In [2]:
P = load_par_file('PAR_FILES/params_namap.par')

_prec = str(P['precision'].lower()) #Select the precision: 64, 32 or 16B


num_frames, first_frame = P['num_frames'], P['first_frame'] #frames to be loaded
#1 frame is defined as 1 second of integration time. 

#Also need to be implemented. 
telemetry = P['telemetry']

#So far, only 'RA and DEC' is implemented and working.   

if P['input_ctype'] == 'RA and DEC':
    coord1 = str('RA')
    coord2 = str('DEC')
    xystage = False
elif P['input_ctype'] == 'AZ and EL':
    coord1 = str('AZ')
    coord2 = str('EL')
    xystage = False
elif P['input_ctype'] == 'CROSS-EL and EL':
    coord1 = str('xEL')
    coord2 = str('EL')
    xystage = False
elif P['input_ctype'] == 'XY Stage':
    coord1 = str('X')
    coord2 = str('Y')
    xystage = True


#Cleaning data parameters <-- needs to be tuned with real data
highpassfreq = P['highpassfreq']
polynomialorder = P['polynomialorder']
despike_bool = P['despike']
sigma,prominence = P['sigma'],P['prominence']
sigma_clipping_bool = ['sigma_clipping']
low_thresh, high_thresh = P['low_thresh'], P['high_thresh'] 
#Beam convolution parameters
convolution, std = P['gaussian_convolution'], P['std'] 
#---------------------------------


In [3]:
dtype_map = {
    '16': np.float16, 'float16': np.float16, 'half': np.float16,
    '32': np.float32, 'float32': np.float32, 'single': np.float32,
    '64': np.float64, 'float64': np.float64, 'double': np.float64
}

int_map = {
    '16': np.int16, 'float16': np.int16, 'half': np.int16,
    '32': np.int32, 'float32': np.int32, 'single': np.int32,
    '64': np.int64, 'float64': np.int64, 'double': np.int64
}

try:
    DT = dtype_map[_prec]
    IT = int_map[_prec]
except KeyError:
    raise ValueError(f"Unsupported precision '{_prec}'. Choose float16/32/64 or 16/32/64.")
print(f"Using numeric dtype: {DT}")

Using numeric dtype: <class 'numpy.float16'>


In [4]:
#Load the name of the KIDs to be analyse. 
filepath = P['hdf5_file']
btable = tb.Table.read(P['detector_table'], format='ascii.tab')

#If an electromagnetic frequency list is provided, then load all the detectors sensitive to them. 
if P['frequencies'] is not None:
    filtered = btable[np.isin(btable['Frequency'], P['frequencies'])]

#If a list of KID name is provided, load them
if P['detectors_to_use'] is not None:
    good_kid_table = tb.Table.read(P['detectors_to_use'], format='ascii.tab')
    filtered = btable[np.isin(btable['Name'], good_kid_table['Name'])]
if P['frequencies'] is None and P['detectors_to_use'] is None:
    filtered = btable
#option in the par file to good kids list

kid_num = Table(filtered)
#if(nbdets is not None): kid_num = filtered['Name'][:nbdets]
print('Nb dets: ', len(kid_num))

Nb dets:  128


In [7]:
#load the boresight offsets in CROSS-EL and EL of the selected detectors. 
dettable = ld.det_table(kid_num, P['detector_table']) 
det_off, _,_ = dettable.loadtable() #noise_det, resp

In [8]:
#load the timestreams and the coordinates. 
#----------------------------------
#Load the data
dataload = ld.data_value(filepath, kid_num, coord1, coord2, first_frame, num_frames,  DT, IT)
det_data, coord1_data, coord2_data, lst_data, lat_data, spf_data, spf_coord, lat_spf = dataload.values()
#-------------------------------

plt.plot(det_data[0])

KeyError: "Unable to synchronously open object (object 'kid_   Name   Resp. WhiteNoise time offset   EL    XEL   Frequency\n--------- ----- ---------- ----------- ------ ------ ---------\nA_01_0006   1.0        0.0         0.0 -0.464 -0.033     715.0_roach' doesn't exist)"